# ASCENT-G H2 Retention Eval — Math & Code Batch

**Experiment:** H2 — Do Phase 1 LoRA adapters (trained on Instruct) transfer to the Base model?

**Tasks:** GSM8K, SVAMP, HumanEval, MBPP

**Protocol:**
1. Load adapter on `Qwen2.5-1.5B-Instruct` → greedy eval on 200 test examples → `instruct_acc`
2. Load same adapter on `Qwen2.5-1.5B` (base) with Instruct tokenizer → same 200 examples → `base_acc`
3. `retention = base_acc / instruct_acc`

**Decision criteria:** Pass if retention > 0.70, Fail if < 0.30

**Adapters:** `/kaggle/input/ascent-g-phase1-adapters/{task_name}/adapter/`

**GPU:** T4 (fp16, autocast_adapter_dtype=False)

**Note:** HumanEval and MBPP use subprocess code execution with 10s timeout per example.

In [ ]:
import subprocess, sys, torch

torch_ver = torch.__version__
torch_base = torch_ver.split("+")[0]

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torchao>=0.16.0", "--no-deps"], check=True)

constraints = (
    f"torch=={torch_base}\n"
    "torchvision\n"
    "torchaudio\n"
    "torchao\n"
).encode()
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "trl==0.17.0", "transformers==4.51.3",
                "peft==0.14.0", "accelerate", "datasets",
                "--constraint", "/dev/stdin"],
               input=constraints, check=True)

import importlib
for mod in ["trl", "transformers", "peft", "datasets"]:
    importlib.invalidate_caches()

import trl, transformers, peft, datasets
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected — abort"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_supported = torch.cuda.is_bf16_supported()

print(f"GPU model  : {gpu_name}")
print(f"VRAM       : {vram_gb:.1f} GB")
print(f"bf16       : {'supported' if bf16_supported else 'NOT supported'}")
print(f"torch      : {torch.__version__}")

# P100 (sm_60) is incompatible with torch 2.10+cu128 — require T4
if "P100" in gpu_name or "sm_60" in str(torch.cuda.get_device_capability()):
    raise RuntimeError(
        f"P100 (sm_60) is incompatible with torch 2.10+cu128. "
        f"Go to Session options → Accelerator → GPU T4 x1 and restart."
    )

MODEL_DTYPE = torch.bfloat16 if bf16_supported else torch.float16
HW_META = {
    "gpu_model": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "bf16_supported": bf16_supported,
    "torch_version": torch.__version__,
}
print("\nHW_META:", HW_META)
print(f"MODEL_DTYPE: {MODEL_DTYPE}")


In [ ]:
from pathlib import Path

ADAPTER_ROOT = Path('/kaggle/input/datasets/chson0316/ascent-g-phase1-adapters')

# Verify adapter files are accessible
cfg_files = sorted(ADAPTER_ROOT.rglob('adapter_config.json'))
print(f'Found {len(cfg_files)} adapter(s):')
for p in cfg_files:
    print(f'  {p.parent}')


In [ ]:
import re, gc, json, datetime, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset

# ── Constants ──────────────────────────────────────────────────────────────
INSTRUCT_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_MODEL_ID     = "Qwen/Qwen2.5-1.5B"
TOKENIZER_ID      = "Qwen/Qwen2.5-1.5B-Instruct"  # used for BOTH models

ADAPTER_BASE = str(ADAPTER_ROOT)
N_EVAL = 200
MAX_NEW_TOKENS_MATH = 256
MAX_NEW_TOKENS_CODE = 256
CODE_EXEC_TIMEOUT   = 10  # seconds per example

MATH_SYSTEM_PROMPT = (
    "Solve the math problem step by step. End with 'The answer is <number>.'"
)
CODE_SYSTEM_PROMPT = (
    "You are a coding assistant. "
    "Write correct Python code and make the final answer a runnable code block."
)

# ── Shared tokenizer (loaded once) ─────────────────────────────────────────
print(f"Loading tokenizer from {TOKENIZER_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)
print("Tokenizer loaded.")


# ── Model load/unload helpers ──────────────────────────────────────────────
def load_model_with_adapter(model_id, adapter_path):
    """Load a causal LM and attach a LoRA adapter. Returns PeftModel."""
    print(f"  Loading base: {model_id}")
    base = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=MODEL_DTYPE,
        device_map="auto",
    )
    print(f"  Attaching adapter: {adapter_path}")
    model = PeftModel.from_pretrained(base, adapter_path, autocast_adapter_dtype=False)
    model.eval()
    return model


def unload_model(model):
    """Delete model and free GPU memory."""
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("  Model unloaded, cache cleared.")


# ── Math helpers ───────────────────────────────────────────────────────────
def extract_final_number(text):
    """Extract the last number (int or float) from a math completion."""
    matches = re.findall(r"[-+]?\d+(?:\.\d+)?", text.replace(",", ""))
    return matches[-1] if matches else None


def greedy_eval_math(model, prompts, answers):
    """
    Greedy math evaluation. Compares last extracted number.
    Returns accuracy (float).
    """
    correct = 0
    for prompt, gold in zip(prompts, answers):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS_MATH,
                do_sample=False,
            )
        completion = tokenizer.decode(
            output[0][input_len:], skip_special_tokens=True
        )
        pred = extract_final_number(completion)
        gold_num = extract_final_number(str(gold))
        if pred is not None and gold_num is not None and pred == gold_num:
            correct += 1
    return correct / len(prompts) if prompts else 0.0


# ── Code helpers ───────────────────────────────────────────────────────────
def extract_code_block(text):
    """Extract the last fenced code block if present, else return raw text."""
    matches = re.findall(r"```(?:python)?\n(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    return matches[-1] if matches else text


def normalize_code_text(text):
    code = extract_code_block(text)
    lines = []
    for raw_line in code.splitlines():
        stripped = raw_line.rstrip()
        if not stripped.strip() or stripped.strip().startswith("#"):
            continue
        lines.append(stripped)
    return "\n".join(lines)


def resolve_humaneval_candidate(raw_prompt, completion, entry_point):
    candidate_code = normalize_code_text(completion)
    if not candidate_code:
        return ""
    if f"def {entry_point}(" in candidate_code:
        return candidate_code
    return str(raw_prompt) + candidate_code


def run_code_check(program, test_code, entry_point=None, timeout_sec=10):
    """
    Run program + test_code in a subprocess with timeout.
    For HumanEval: entry_point required; test_code calls check(fn).
    For MBPP: test_code is plain assert statements.
    Returns (passed: bool, timed_out: bool).
    """
    payload = json.dumps({
        "program": program,
        "test_code": str(test_code),
        "entry_point": str(entry_point) if entry_point else "",
    }, ensure_ascii=False)

    runner = '''
import json, sys
payload = json.loads(sys.argv[1])
namespace = {}
exec(payload["program"], namespace)
exec(payload["test_code"], namespace)
ep = payload["entry_point"].strip()
if ep:
    if ep not in namespace:
        raise NameError(f"entry_point '{ep}' not defined")
    check_fn = namespace.get("check")
    if not callable(check_fn):
        raise NameError("HumanEval test block did not define callable 'check'")
    check_fn(namespace[ep])
'''
    try:
        completed = subprocess.run(
            [sys.executable, "-c", runner, payload],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=timeout_sec,
            check=False,
        )
    except subprocess.TimeoutExpired:
        return False, True
    return completed.returncode == 0, False


def greedy_eval_code_humaneval(model, examples):
    """
    Greedy HumanEval evaluation with subprocess code execution.
    examples: list of dicts with keys: prompt, raw_prompt, test, entry_point
    Returns (accuracy, n_timeouts).
    """
    correct = 0
    timeouts = 0
    for ex in examples:
        inputs = tokenizer(ex["prompt"], return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS_CODE,
                do_sample=False,
            )
        completion = tokenizer.decode(
            output[0][input_len:], skip_special_tokens=True
        )
        candidate = resolve_humaneval_candidate(
            ex["raw_prompt"], completion, ex["entry_point"]
        )
        if not candidate:
            continue
        passed, timed_out = run_code_check(
            candidate, ex["test"], entry_point=ex["entry_point"],
            timeout_sec=CODE_EXEC_TIMEOUT,
        )
        if timed_out:
            timeouts += 1
        if passed:
            correct += 1
    acc = correct / len(examples) if examples else 0.0
    return acc, timeouts


def greedy_eval_code_mbpp(model, examples):
    """
    Greedy MBPP evaluation with subprocess code execution.
    examples: list of dicts with keys: prompt, test_list (list of assert strs)
    Returns (accuracy, n_timeouts).
    """
    correct = 0
    timeouts = 0
    for ex in examples:
        inputs = tokenizer(ex["prompt"], return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS_CODE,
                do_sample=False,
            )
        completion = tokenizer.decode(
            output[0][input_len:], skip_special_tokens=True
        )
        code = normalize_code_text(completion)
        test_code = "\n".join(ex["test_list"])
        passed, timed_out = run_code_check(
            code, test_code, entry_point=None,
            timeout_sec=CODE_EXEC_TIMEOUT,
        )
        if timed_out:
            timeouts += 1
        if passed:
            correct += 1
    acc = correct / len(examples) if examples else 0.0
    return acc, timeouts


# ── Retention helper ───────────────────────────────────────────────────────
def compute_retention(base_acc, instruct_acc):
    """Retention = base_acc / instruct_acc. Returns (retention, verdict)."""
    if instruct_acc == 0.0:
        return None, "UNDEFINED (instruct_acc=0)"
    retention = base_acc / instruct_acc
    if retention >= 0.70:
        verdict = "PASS"
    elif retention < 0.30:
        verdict = "FAIL"
    else:
        verdict = "MARGINAL"
    return retention, verdict


print("Helper functions defined.")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 1: GSM8K
# Phase 1 instruct best_reward: 0.9125
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "gsm8k"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"
PHASE1_BEST_REWARD = 0.9125

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Phase 1 best_reward (instruct): {PHASE1_BEST_REWARD}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: gsm8k, main, test ...")
ds = load_dataset("gsm8k", "main", split="test")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

def format_gsm8k(example):
    question = str(example["question"])
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": MATH_SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    # GSM8K answer format: "...#### 42"
    answer = str(example["answer"])
    return {"prompt": prompt, "answer": answer}

ds_fmt = ds.map(format_gsm8k)
prompts = ds_fmt["prompt"]
answers = ds_fmt["answer"]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
gsm8k_instruct_acc = greedy_eval_math(model, prompts, answers)
print(f"  instruct_acc = {gsm8k_instruct_acc:.4f}")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
gsm8k_base_acc = greedy_eval_math(model, prompts, answers)
print(f"  base_acc = {gsm8k_base_acc:.4f}")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
gsm8k_retention, gsm8k_verdict = compute_retention(gsm8k_base_acc, gsm8k_instruct_acc)
print(f"\n{'─'*40}")
print(f"GSM8K  instruct_acc : {gsm8k_instruct_acc:.4f}")
print(f"GSM8K  base_acc     : {gsm8k_base_acc:.4f}")
print(f"GSM8K  retention    : {gsm8k_retention:.4f}")
print(f"GSM8K  verdict      : {gsm8k_verdict}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 2: SVAMP
# Phase 1 instruct best_reward: 0.9500
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "svamp"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"
PHASE1_BEST_REWARD = 0.9500

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Phase 1 best_reward (instruct): {PHASE1_BEST_REWARD}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: ChilleD/SVAMP, test ...")
ds = load_dataset("ChilleD/SVAMP", split="test")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

def format_svamp(example):
    question = str(example["Body"]) + " " + str(example["Question"])
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": MATH_SYSTEM_PROMPT},
            {"role": "user",   "content": question},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {"prompt": prompt, "answer": str(example["Answer"])}

ds_fmt = ds.map(format_svamp)
prompts = ds_fmt["prompt"]
answers = ds_fmt["answer"]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
svamp_instruct_acc = greedy_eval_math(model, prompts, answers)
print(f"  instruct_acc = {svamp_instruct_acc:.4f}")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
svamp_base_acc = greedy_eval_math(model, prompts, answers)
print(f"  base_acc = {svamp_base_acc:.4f}")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
svamp_retention, svamp_verdict = compute_retention(svamp_base_acc, svamp_instruct_acc)
print(f"\n{'─'*40}")
print(f"SVAMP  instruct_acc : {svamp_instruct_acc:.4f}")
print(f"SVAMP  base_acc     : {svamp_base_acc:.4f}")
print(f"SVAMP  retention    : {svamp_retention:.4f}")
print(f"SVAMP  verdict      : {svamp_verdict}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 3: HumanEval
# Phase 1 instruct best_reward: 0.7125
# Note: code execution via subprocess, 10s timeout per example
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "humaneval"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"
PHASE1_BEST_REWARD = 0.7125

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Phase 1 best_reward (instruct): {PHASE1_BEST_REWARD}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"Code exec timeout: {CODE_EXEC_TIMEOUT}s per example")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: openai/openai_humaneval, test ...")
ds = load_dataset("openai/openai_humaneval", split="test")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

def format_humaneval(example):
    raw_prompt = str(example["prompt"])
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": CODE_SYSTEM_PROMPT},
            {"role": "user",   "content": raw_prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": prompt,
        "raw_prompt": raw_prompt,
        "test": str(example["test"]),
        "entry_point": str(example["entry_point"]),
        "answer": str(example["canonical_solution"]),
    }

ds_fmt = ds.map(format_humaneval)
he_examples = [
    {
        "prompt": ex["prompt"],
        "raw_prompt": ex["raw_prompt"],
        "test": ex["test"],
        "entry_point": ex["entry_point"],
    }
    for ex in ds_fmt
]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval (with code execution) ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
t0 = time.time()
humaneval_instruct_acc, humaneval_instruct_timeouts = greedy_eval_code_humaneval(model, he_examples)
elapsed = time.time() - t0
print(f"  instruct_acc = {humaneval_instruct_acc:.4f}  timeouts={humaneval_instruct_timeouts}  elapsed={elapsed:.1f}s")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval (with code execution) ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
t0 = time.time()
humaneval_base_acc, humaneval_base_timeouts = greedy_eval_code_humaneval(model, he_examples)
elapsed = time.time() - t0
print(f"  base_acc = {humaneval_base_acc:.4f}  timeouts={humaneval_base_timeouts}  elapsed={elapsed:.1f}s")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
humaneval_retention, humaneval_verdict = compute_retention(humaneval_base_acc, humaneval_instruct_acc)
print(f"\n{'─'*40}")
print(f"HumanEval  instruct_acc : {humaneval_instruct_acc:.4f}  (timeouts={humaneval_instruct_timeouts})")
print(f"HumanEval  base_acc     : {humaneval_base_acc:.4f}  (timeouts={humaneval_base_timeouts})")
print(f"HumanEval  retention    : {humaneval_retention:.4f}")
print(f"HumanEval  verdict      : {humaneval_verdict}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# Task 4: MBPP
# Phase 1 instruct best_reward: 0.6750
# Note: code execution via subprocess, 10s timeout per example
# ════════════════════════════════════════════════════════════════
TASK_NAME    = "mbpp"
ADAPTER_PATH = f"{ADAPTER_BASE}/{TASK_NAME}/adapter"
PHASE1_BEST_REWARD = 0.6750

print(f"\n{'='*60}")
print(f"TASK: {TASK_NAME.upper()}")
print(f"Phase 1 best_reward (instruct): {PHASE1_BEST_REWARD}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"Code exec timeout: {CODE_EXEC_TIMEOUT}s per example")
print(f"{'='*60}")

# Load dataset
print("Loading dataset: google-research-datasets/mbpp, test ...")
ds = load_dataset("google-research-datasets/mbpp", split="test")
ds = ds.shuffle(seed=42).select(range(min(N_EVAL, len(ds))))
print(f"Eval examples: {len(ds)}")

def format_mbpp(example):
    task_description = str(example["text"])
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": CODE_SYSTEM_PROMPT},
            {"role": "user",   "content": task_description},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    return {
        "prompt": prompt,
        "test_list": example["test_list"],
        "answer": str(example["code"]),
    }

ds_fmt = ds.map(format_mbpp)
mbpp_examples = [
    {"prompt": ex["prompt"], "test_list": ex["test_list"]}
    for ex in ds_fmt
]

# ── Instruct eval ──────────────────────────────────────────────
print("\n[1/2] Instruct model eval (with code execution) ...")
model = load_model_with_adapter(INSTRUCT_MODEL_ID, ADAPTER_PATH)
t0 = time.time()
mbpp_instruct_acc, mbpp_instruct_timeouts = greedy_eval_code_mbpp(model, mbpp_examples)
elapsed = time.time() - t0
print(f"  instruct_acc = {mbpp_instruct_acc:.4f}  timeouts={mbpp_instruct_timeouts}  elapsed={elapsed:.1f}s")
unload_model(model)

# ── Base eval ──────────────────────────────────────────────────
print("\n[2/2] Base model eval (with code execution) ...")
model = load_model_with_adapter(BASE_MODEL_ID, ADAPTER_PATH)
t0 = time.time()
mbpp_base_acc, mbpp_base_timeouts = greedy_eval_code_mbpp(model, mbpp_examples)
elapsed = time.time() - t0
print(f"  base_acc = {mbpp_base_acc:.4f}  timeouts={mbpp_base_timeouts}  elapsed={elapsed:.1f}s")
unload_model(model)

# ── Result ─────────────────────────────────────────────────────
mbpp_retention, mbpp_verdict = compute_retention(mbpp_base_acc, mbpp_instruct_acc)
print(f"\n{'─'*40}")
print(f"MBPP  instruct_acc : {mbpp_instruct_acc:.4f}  (timeouts={mbpp_instruct_timeouts})")
print(f"MBPP  base_acc     : {mbpp_base_acc:.4f}  (timeouts={mbpp_base_timeouts})")
print(f"MBPP  retention    : {mbpp_retention:.4f}")
print(f"MBPP  verdict      : {mbpp_verdict}")

In [ ]:
import json, datetime
from pathlib import Path

results = {
    "experiment": "H2-retention-math-code",
    "date": datetime.datetime.utcnow().isoformat(),
    "hardware": HW_META,
    "protocol": {
        "n_eval": N_EVAL,
        "max_new_tokens_math": MAX_NEW_TOKENS_MATH,
        "max_new_tokens_code": MAX_NEW_TOKENS_CODE,
        "code_exec_timeout_sec": CODE_EXEC_TIMEOUT,
        "decoding": "greedy",
        "instruct_model": INSTRUCT_MODEL_ID,
        "base_model": BASE_MODEL_ID,
        "tokenizer": TOKENIZER_ID,
        "pass_threshold": 0.70,
        "fail_threshold": 0.30,
    },
    "tasks": {
        "gsm8k": {
            "phase1_best_reward": 0.9125,
            "instruct_acc": gsm8k_instruct_acc,
            "base_acc": gsm8k_base_acc,
            "retention": gsm8k_retention,
            "verdict": gsm8k_verdict,
        },
        "svamp": {
            "phase1_best_reward": 0.9500,
            "instruct_acc": svamp_instruct_acc,
            "base_acc": svamp_base_acc,
            "retention": svamp_retention,
            "verdict": svamp_verdict,
        },
        "humaneval": {
            "phase1_best_reward": 0.7125,
            "instruct_acc": humaneval_instruct_acc,
            "base_acc": humaneval_base_acc,
            "instruct_timeouts": humaneval_instruct_timeouts,
            "base_timeouts": humaneval_base_timeouts,
            "retention": humaneval_retention,
            "verdict": humaneval_verdict,
        },
        "mbpp": {
            "phase1_best_reward": 0.6750,
            "instruct_acc": mbpp_instruct_acc,
            "base_acc": mbpp_base_acc,
            "instruct_timeouts": mbpp_instruct_timeouts,
            "base_timeouts": mbpp_base_timeouts,
            "retention": mbpp_retention,
            "verdict": mbpp_verdict,
        },
    },
}

output_path = Path("/kaggle/working/h2_retention_results_math_code.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*60)
print("H2 RETENTION RESULTS — MATH & CODE BATCH")
print("="*60)
for task, v in results["tasks"].items():
    ret = v['retention']
    ret_str = f"{ret:.4f}" if ret is not None else "N/A"
    timeout_str = ""
    if "instruct_timeouts" in v:
        timeout_str = f"  timeouts(i/b)={v['instruct_timeouts']}/{v['base_timeouts']}"
    print(f"  {task:<14} instruct={v['instruct_acc']:.4f}  base={v['base_acc']:.4f}  "
          f"retention={ret_str}  [{v['verdict']}]{timeout_str}")
print("="*60)
print(f"\nResults saved to: {output_path}")
print(json.dumps(results, indent=2))